# `test_preprocessing.py` — Unit Tests for `DataPreprocessor`

## Purpose

Validates the full preprocessing pipeline that transforms raw HR data into model-ready
train/test arrays. Tests cover feature/target separation, one-hot encoding, stratified
splitting, SMOTE oversampling, and the end-to-end pipeline method.

---

## Module Under Test

`src.preprocessing.data_preprocessor.DataPreprocessor`

---

## Test Classes at a Glance

| Class | Methods Tested | What It Verifies |
|-------|----------------|-----------------|
| `TestSeparateFeaturesTarget` | `separate_features_target()` | Target excluded from X, y name/values, column count, missing target error |
| `TestEncodeCategorical` | `encode_categorical()` | Categorical columns removed, dummy columns created, new object returned |
| `TestStratifiedSplit` | `stratified_split()` | Train+test = full dataset, class distribution preserved |
| `TestApplySmote` | `apply_smote()` | Classes balanced after SMOTE, column names preserved |
| `TestRunPreprocessingPipeline` | `run_preprocessing_pipeline()` | Returns 4 elements, correct types |

---

## Fixtures Used

| Fixture | Source |
|---------|--------|
| `sample_df` | `conftest.py` |

---

## How to Run

```bash
pytest tests/test_preprocessing.py -v
```


---

## `TestSeparateFeaturesTarget`

**Purpose:** Tests `separate_features_target()` — verifies that the `left` column is removed
from the feature matrix and returned as a separate Series with binary values only.

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_target_not_in_X` | `left` column absent from `X` |
| `test_target_in_y` | `y.name == "left"` and values are `{0, 1}` only |
| `test_X_has_remaining_columns` | `X` has exactly `len(sample_df.columns) - 1` columns |
| `test_missing_target_raises` | Raises `PreprocessingError` when target column is absent |


In [ ]:
import pandas as pd
import pytest

from src.preprocessing.data_preprocessor import DataPreprocessor
from src.utils.exceptions import PreprocessingError


class TestSeparateFeaturesTarget:
    def test_target_not_in_X(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, y = preprocessor.separate_features_target()
        assert "left" not in X.columns

    def test_target_in_y(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, y = preprocessor.separate_features_target()
        assert y.name == "left"
        assert set(y.unique()).issubset({0, 1})

    def test_X_has_remaining_columns(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, y = preprocessor.separate_features_target()
        assert len(X.columns) == len(sample_df.columns) - 1

    def test_missing_target_raises(self, sample_df):
        df_no_target = sample_df.drop(columns=["left"])
        preprocessor = DataPreprocessor(df_no_target, target_col="left")
        with pytest.raises(PreprocessingError):
            preprocessor.separate_features_target()


---

## `TestEncodeCategorical`

**Purpose:** Tests `encode_categorical()` — verifies that `sales` and `salary` are replaced by
one-hot dummy columns, and that the method returns a new DataFrame (not mutating the input).

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_no_sales_or_salary_in_output` | Neither `sales` nor `salary` column appears in result |
| `test_dummy_columns_created` | At least 2 `salary_*` columns are present after encoding |
| `test_output_is_new_dataframe` | Returned object is not the same reference as the input |


In [ ]:
class TestEncodeCategorical:
    def test_no_sales_or_salary_in_output(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, _ = preprocessor.separate_features_target()
        X_enc = preprocessor.encode_categorical(X)
        assert "sales" not in X_enc.columns
        assert "salary" not in X_enc.columns

    def test_dummy_columns_created(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, _ = preprocessor.separate_features_target()
        X_enc = preprocessor.encode_categorical(X)
        salary_cols = [c for c in X_enc.columns if c.startswith("salary_")]
        assert len(salary_cols) >= 2

    def test_output_is_new_dataframe(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, _ = preprocessor.separate_features_target()
        X_enc = preprocessor.encode_categorical(X)
        assert X_enc is not X


---

## `TestStratifiedSplit`

**Purpose:** Tests `stratified_split()` — verifies that the split sizes are correct and that
class proportions in train and test sets closely mirror the original distribution (stratification).

### Test Methods

| Method | Verifies |
|--------|----------|
| `test_train_test_sizes` | `len(X_train) + len(X_test) == len(sample_df)` and test ratio ≈ 0.20 |
| `test_class_distribution_preserved` | Turnover rate in train/test within 5% of original |


In [ ]:
class TestStratifiedSplit:
    def test_train_test_sizes(self, sample_df):
        preprocessor = DataPreprocessor(sample_df, test_size=0.2)
        X, y = preprocessor.separate_features_target()
        X_enc = preprocessor.encode_categorical(X)
        X_train, X_test, y_train, y_test = preprocessor.stratified_split(X_enc, y)
        total = len(X_train) + len(X_test)
        assert total == len(sample_df)
        assert abs(len(X_test) / total - 0.2) < 0.05

    def test_class_distribution_preserved(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, y = preprocessor.separate_features_target()
        X_enc = preprocessor.encode_categorical(X)
        _, _, y_train, y_test = preprocessor.stratified_split(X_enc, y)
        original_rate = y.mean()
        train_rate = y_train.mean()
        test_rate = y_test.mean()
        assert abs(train_rate - original_rate) < 0.05
        assert abs(test_rate - original_rate) < 0.05


---

## `TestApplySmote` · `TestRunPreprocessingPipeline`

**Purpose:**
- `TestApplySmote` verifies SMOTE oversamples the minority class until both classes are equal,
  while preserving all feature column names.
- `TestRunPreprocessingPipeline` verifies the convenience method returns exactly 4 elements
  with the correct types (`pd.DataFrame`, `pd.DataFrame`, `pd.Series`, `pd.Series`).

### Test Methods

| Method | Class | Verifies |
|--------|-------|----------|
| `test_smote_balances_classes` | `TestApplySmote` | `counts[0] == counts[1]` after SMOTE |
| `test_smote_preserves_columns` | `TestApplySmote` | Column list unchanged |
| `test_returns_four_elements` | `TestRunPreprocessingPipeline` | `len(result) == 4` |
| `test_output_types` | `TestRunPreprocessingPipeline` | X types are `DataFrame`, y types are `Series` |


In [ ]:
class TestApplySmote:
    def test_smote_balances_classes(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, y = preprocessor.separate_features_target()
        X_enc = preprocessor.encode_categorical(X)
        X_train, _, y_train, _ = preprocessor.stratified_split(X_enc, y)
        X_res, y_res = preprocessor.apply_smote(X_train, y_train)
        counts = y_res.value_counts()
        assert counts[0] == counts[1]

    def test_smote_preserves_columns(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X, y = preprocessor.separate_features_target()
        X_enc = preprocessor.encode_categorical(X)
        X_train, _, y_train, _ = preprocessor.stratified_split(X_enc, y)
        X_res, _ = preprocessor.apply_smote(X_train, y_train)
        assert list(X_res.columns) == list(X_train.columns)


class TestRunPreprocessingPipeline:
    def test_returns_four_elements(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        result = preprocessor.run_preprocessing_pipeline()
        assert len(result) == 4

    def test_output_types(self, sample_df):
        preprocessor = DataPreprocessor(sample_df)
        X_train, X_test, y_train, y_test = preprocessor.run_preprocessing_pipeline()
        assert isinstance(X_train, pd.DataFrame)
        assert isinstance(X_test, pd.DataFrame)
        assert isinstance(y_train, pd.Series)
        assert isinstance(y_test, pd.Series)
